# STAIR Baseline Reproduction Experiment

**Mục tiêu:** Tái tạo kết quả bảng Table 2 trong paper STAIR (SIGIR 2025)  
**Môi trường:** Kaggle Notebook — GPU T4 / P100, 16 GB VRAM  
**Datasets:** Amazon2014Baby_550_MMRec, Amazon2014Sports_550_MMRec, Amazon2014Electronics_550_MMRec  
**Lưu ý:** Do giới hạn GPU-hour Kaggle, mỗi dataset chạy 1 lần (1 seed) thay vì 5 lần như paper.  
Kết quả được đối chiếu theo numerical fidelity (% lệch so với Table 2).


In [ ]:
# ================================================================
# CELL 1: Environment Setup
# Audit fixes:
#   - Pin freerec==0.8.5 (env.sh gốc) thay vì 0.8.3 / 0.9.5
#   - Pin torchdata==0.7.1 theo env.sh gốc
#   - PyG cài qua wheel index chính thức tránh CUDA mismatch
#   - Dùng subprocess.run + check=True, không dùng os.system
# ================================================================
import os, shutil, subprocess, sys, time

os.chdir('/kaggle/working')
if os.path.exists('STAIR'):
    shutil.rmtree('STAIR')

print('🔄 Cloning STAIR repository...')
subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/yhhe2004/STAIR.git'],
    check=True
)

# Pin đúng version gốc theo env.sh của tác giả
print('🔄 Installing freerec==0.8.5 & torchdata==0.7.1...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'freerec==0.8.5', 'torchdata==0.7.1', 'nvidia-ml-py'],
    check=True
)

# PyG: dùng wheel index khớp torch/cuda có sẵn trên Kaggle
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'🔄 Installing torch-geometric for torch={TORCH_VER}+{CUDA_TAG}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch-geometric',
     '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'],
    check=True
)

print('\n✅ Environment setup complete!')
print(f'   torch={torch.__version__}, cuda_available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')


In [ ]:
# ================================================================
# CELL 2: Data Preparation
# Audit fixes:
#   - DATA_ROOT = '/kaggle/working/STAIR/data' (thư mục data/ chuẩn của repo)
#   - Dataset name đúng: Amazon2014Baby_550_MMRec (không phải 'baby')
#   - Kiểm tra required files sau khi copy
#   - Validate .pkl files cho multimodal features
# ================================================================
import os, shutil

DATA_ROOT = '/kaggle/working/STAIR/data'
os.makedirs(DATA_ROOT, exist_ok=True)

DATASET_MAP = {
    'Amazon2014Baby_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Baby_550_MMRec',
    'Amazon2014Sports_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Sports_550_MMRec',
    'Amazon2014Electronics_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Electronics_550_MMRec',
}

REQUIRED_FILES = {
    'train.txt', 'valid.txt', 'test.txt',
    'textual_modality.pkl', 'visual_modality.pkl'
}

for ds_name, src in DATASET_MAP.items():
    dest = os.path.join(DATA_ROOT, ds_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    present = set(os.listdir(dest))
    missing = REQUIRED_FILES - present
    if missing:
        raise FileNotFoundError(f'[{ds_name}] Missing files: {missing}')
    sizes = {f: f'{os.path.getsize(os.path.join(dest,f))/1024/1024:.1f}MB' for f in sorted(present)}
    print(f'✅ {ds_name}')
    for fname, sz in sizes.items():
        print(f'   {fname}: {sz}')

print(f'\n✅ All datasets ready in {DATA_ROOT}')


In [ ]:
# ================================================================
# CELL 3: Smoke Test (1 epoch, Baby only)
# Chạy 1 epoch Baby trước để xác nhận:
#   - Dataset path resolve đúng
#   - freerec API không có breaking change
#   - Không có lỗi CUDA/PyG
# Tiết kiệm GPU-hour trước khi chạy full 500 epochs
# ================================================================
import subprocess, os

os.makedirs('/kaggle/working/logs', exist_ok=True)

print('🧪 Running smoke test: Baby, 1 epoch...')
result = subprocess.run(
    ['python', 'main.py',
     '--dataset',      'Amazon2014Baby_550_MMRec',
     '--root',         '/kaggle/working/STAIR/data',
     '--weight-decay', '0.3',
     '--gamma',        '0.1',
     '--batch-size',   '1024',
     '--epochs',       '1',
     '--eval-freq',    '1',
    ],
    capture_output=True, text=True,
    cwd='/kaggle/working/STAIR'
)

if result.returncode != 0:
    print('❌ Smoke test FAILED. stderr:')
    print(result.stderr[-3000:])
    raise RuntimeError('Smoke test failed — do NOT proceed to full training!')
else:
    print('✅ Smoke test passed! Proceeding to full training...')
    print(result.stdout[-800:])


In [ ]:
# ================================================================
# CELL 4: Full Training — 500 epochs x 3 datasets
# Audit fixes:
#   - Log stream ra file thay vì capture_output (tránh OOM RAM)
#   - returncode check sau mỗi run → không im lặng khi crash
#   - Dataset name đúng full: Amazon2014*_550_MMRec
#   - Hyperparams từ paper Table 2 (weight-decay, gamma, batch-size)
#   - eval-freq=5 để log đủ dày cho learning curve
# ================================================================
import subprocess, os, time, pynvml, threading

os.makedirs('/kaggle/working/logs', exist_ok=True)

# VRAM profiler
all_logs = {
    'baby':        {'vram': [], 'time': []},
    'sports':      {'vram': [], 'time': []},
    'electronics': {'vram': [], 'time': []},
}
profiling_active = False
current_ds_key   = None

def hardware_profiler(interval=1.0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    t0 = time.time()
    while profiling_active:
        mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if current_ds_key:
            all_logs[current_ds_key]['vram'].append(mem.used / 1024**2)
            all_logs[current_ds_key]['time'].append(time.time() - t0)
        time.sleep(interval)
    pynvml.nvmlShutdown()

# Hyperparams khớp đúng paper Table 2 + env.sh gốc tác giả
RUNS = [
    dict(key='baby',        dataset='Amazon2014Baby_550_MMRec',        weight_decay='0.3', gamma='0.1', batch_size='1024'),
    dict(key='sports',      dataset='Amazon2014Sports_550_MMRec',      weight_decay='0.1', gamma='0.2', batch_size='1024'),
    dict(key='electronics', dataset='Amazon2014Electronics_550_MMRec', weight_decay='0.1', gamma='0.4', batch_size='4096'),
]

for run in RUNS:
    key      = run['key']
    log_path = f'/kaggle/working/logs/{key}.log'
    print(f"\n{'='*55}")
    print(f"🚀 TRAINING: {key.upper()} | log → {log_path}")
    print('='*55)

    current_ds_key   = key
    profiling_active = True
    prof_thread = threading.Thread(target=hardware_profiler, args=(1.0,), daemon=True)
    prof_thread.start()

    t0 = time.time()
    # Stream log ra file thay vì capture_output=True (tránh OOM với Electronics ~500ep)
    with open(log_path, 'w', encoding='utf-8') as logf:
        result = subprocess.run(
            ['python', 'main.py',
             '--dataset',      run['dataset'],
             '--root',         '/kaggle/working/STAIR/data',
             '--weight-decay', run['weight_decay'],
             '--gamma',        run['gamma'],
             '--batch-size',   run['batch_size'],
             '--epochs',       '500',
             '--eval-freq',    '5',
            ],
            stdout=logf, stderr=subprocess.STDOUT,
            cwd='/kaggle/working/STAIR'
        )

    elapsed = time.time() - t0
    profiling_active = False
    prof_thread.join(timeout=5)

    if result.returncode != 0:
        print(f'❌ {key.upper()} FAILED (rc={result.returncode}) after {elapsed/60:.1f}min')
        print(f'Last 30 lines of {log_path}:')
        with open(log_path, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        print(''.join(lines[-30:]))
    else:
        print(f'✅ {key.upper()} done in {elapsed/60:.1f} min | log: {log_path}')
        with open(log_path, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        # Print last 15 lines (final metrics)
        print(''.join(lines[-15:]))

print('\n🎉 All training runs completed!')


In [ ]:
# ================================================================
# CELL 5: VRAM Usage Visualisation
# ================================================================
import matplotlib.pyplot as plt

KEYS   = ['baby', 'sports', 'electronics']
COLORS = ['#1f77b4', '#ff7f0e', '#d62728']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GPU VRAM Usage — STAIR Reproduction (Kaggle T4)',
             fontsize=15, fontweight='bold')

for idx, (key, color) in enumerate(zip(KEYS, COLORS)):
    ax = axes[idx // 2, idx % 2]
    times = all_logs[key]['time']
    vrams = all_logs[key]['vram']
    if vrams:
        ax.plot(times, vrams, color=color, linewidth=1.5)
        ax.fill_between(times, vrams, color=color, alpha=0.25)
        ax.axhline(max(vrams), color='black', linestyle='--', linewidth=0.8,
                   label=f'Peak: {max(vrams):.0f} MB')
        ax.legend(fontsize=9)
    ax.set_title(key.capitalize(), fontsize=12)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('VRAM (MB)')
    ax.grid(True, linestyle=':', alpha=0.5)

ax_sum = axes[1, 1]
for key, color in zip(KEYS, COLORS):
    vrams = all_logs[key]['vram']
    if vrams:
        ax_sum.plot(all_logs[key]['time'], vrams, color=color,
                    label=f"{key} (Peak: {max(vrams):.0f} MB)")
ax_sum.set_title('All Datasets Combined', fontsize=12)
ax_sum.set_xlabel('Time (s)')
ax_sum.set_ylabel('VRAM (MB)')
ax_sum.grid(True, linestyle=':', alpha=0.5)
ax_sum.legend(fontsize=9)

plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.savefig('/kaggle/working/vram_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /kaggle/working/vram_profile.png')


In [ ]:
# ================================================================
# CELL 6: Extract & Display Final Metrics from Log Files
# ================================================================
import re, os

METRICS_PATTERN = re.compile(
    r'TEST.*?RECALL@10.*?([0-9.]+).*?RECALL@20.*?([0-9.]+).*?NDCG@10.*?([0-9.]+).*?NDCG@20.*?([0-9.]+)',
    re.IGNORECASE
)

# Paper Table 2 reference values
PAPER_RESULTS = {
    'baby':        dict(r10=0.0652, r20=0.1014, n10=0.0352, n20=0.0443),
    'sports':      dict(r10=0.0758, r20=0.1147, n10=0.0416, n20=0.0519),
    'electronics': dict(r10=0.0476, r20=0.0721, n10=0.0258, n20=0.0322),
}

print(f"{'Dataset':<15} {'Metric':<12} {'Paper':>8} {'Reproduced':>12} {'Delta%':>8}")
print('-' * 60)

for key in ['baby', 'sports', 'electronics']:
    log_path = f'/kaggle/working/logs/{key}.log'
    if not os.path.exists(log_path):
        print(f'{key}: log file not found, skip.')
        continue
    with open(log_path, encoding='utf-8', errors='replace') as f:
        content = f.read()
    # Find last TEST line
    test_lines = [l for l in content.splitlines() if 'TEST' in l and 'RECALL' in l.upper()]
    if not test_lines:
        print(f'{key}: no TEST metrics found in log.')
        continue
    last = test_lines[-1]
    nums = re.findall(r'[0-9]+\.[0-9]+', last)
    if len(nums) >= 4:
        repro = dict(r10=float(nums[0]), r20=float(nums[1]), n10=float(nums[2]), n20=float(nums[3]))
        paper = PAPER_RESULTS[key]
        for metric, p_val, r_val in [
            ('Recall@10', paper['r10'], repro['r10']),
            ('Recall@20', paper['r20'], repro['r20']),
            ('NDCG@10',   paper['n10'], repro['n10']),
            ('NDCG@20',   paper['n20'], repro['n20']),
        ]:
            delta = (r_val - p_val) / p_val * 100
            flag = '⚠️' if abs(delta) > 5 else '✅'
            print(f"{key:<15} {metric:<12} {p_val:>8.4f} {r_val:>12.4f} {delta:>+7.2f}% {flag}")
    else:
        print(f'{key}: could not parse metrics from: {last}')

print('\nNote: Delta > ±5% flagged with ⚠️ — check config alignment with paper yaml.')
